In [22]:
import sys
print(sys.executable)
import yfinance, statsmodels, seaborn

c:\Users\vince\portfolio-sensitivity\.venv\Scripts\python.exe


In [23]:
import yfinance as yf 
import pandas as pd

raw = yf.download(["DE", "WMT", "F", "CMG","NDAQ","SPY","^TNX"], start="2021-07-01", end="2026-07-01")

prices = raw["Close"]
prices.tail()

[*********************100%***********************]  7 of 7 completed


Ticker,CMG,DE,F,NDAQ,SPY,WMT,^TNX
Date,,,,,,,
2026-06-24,31.690001,599.196899,13.84,81.610001,733.239990,119.000000,4.402
2026-06-25,32.279999,629.129333,14.11,77.650002,734.299988,115.779999,4.392
2026-06-26,33.340000,611.654602,14.13,78.559998,728.989990,115.690002,4.372
2026-06-29,32.970001,625.010010,14.02,76.849998,741.000000,114.599998,4.374
2026-06-30,34.000000,634.330017,13.90,78.820000,746.770020,113.260002,4.418


In [25]:
#transforming into the regression
returns = prices[["DE","SPY"]].pct_change() * 100
d_bps = prices["^TNX"].diff() * 100 #daily yield change in bps

df = pd.concat([returns, d_bps.rename("d_bps")], axis = 1).dropna()
df.describe()

,DE,SPY,d_bps
count,1253.000000,1253.000000,1253.000000
mean,0.069006,0.055218,0.234477
std,1.858642,1.082127,6.262050
min,-14.072215,-5.854299,-32.200003
25%,-0.938034,-0.456556,-3.600001
50%,0.056481,0.073105,0.199997
75%,1.073280,0.632841,4.100013
max,11.584948,10.501938,21.000004


In [26]:
import statsmodels.api as sm 

X = sm.add_constant(df[["SPY", "d_bps"]])
model = sm.OLS(df["DE"], X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                     DE   R-squared:                       0.204
Model:                            OLS   Adj. R-squared:                  0.203
Method:                 Least Squares   F-statistic:                     160.4
Date:                Fri, 31 Jul 2026   Prob (F-statistic):           1.02e-62
Time:                        18:53:09   Log-Likelihood:                -2411.0
No. Observations:                1253   AIC:                             4828.
Df Residuals:                    1250   BIC:                             4843.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0222      0.047      0.473      0.6

In [21]:
print(prices["^TNX"].tail())
print(df["d_bps"].std())

Date
2026-06-24    4.402
2026-06-25    4.392
2026-06-26    4.372
2026-06-29    4.374
2026-06-30    4.418
Name: ^TNX, dtype: float64
6.262049872058005
